In [1]:
from pytential import sympy_pytential
from pytential.reduce import min_pytential
import numpy as np
from sympy import log, symbols
import plotly.graph_objects as go

This file demonstrates how to create, combine, and manipulate pytentials.  Equibilriation is used to reduce free variables. 

# Create phases

Create two binary ideal solutions and add the lattice constraint in terms of the phase volume. 

In [2]:
# Assemble sympy expressions.
c0a, c1a, c0b, c1b, Va, Vb = symbols('c0a, c1a, c0b, c1b, Va, Vb')

RT = 8.134*300
fa_sp = c0a*RT*(2+log(c0a/(c0a+c1a))) +c1a*RT*(0+log(c1a/(c0a+c1a)))
fb_sp = c0b*RT*(0+log(c0b/(c0b+c1b))) +c1b*RT*(1+log(c1b/(c0b+c1b)))
lattice_constraint_a = c0a + c1a - Va
lattice_constraint_b = c0b + c1b - Vb

In [3]:
# Assemble pytential with constraint. 
fa = sympy_pytential(fa_sp, constraints_sym=[lattice_constraint_a])
fb = sympy_pytential(fb_sp, constraints_sym=[lattice_constraint_b])

In [4]:
print('fa:', fa)

fa: x = ['Va', 'c0a', 'c1a']

f(x) = 2440.2*c0a*(log(c0a/(c0a + c1a)) + 2) + 2440.2*c1a*log(c1a/(c0a + c1a))

f'(x)= [0, -2440.2*c1a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c0a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 2440.2*log(c0a/(c0a + c1a)) + 4880.4, -2440.2*c0a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c1a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 2440.2*log(c1a/(c0a + c1a))]

f"(x)= [[0, 0, 0], [0, -2440.2*c0a/(c0a + c1a)**2 + 2440.2*c1a/(c0a + c1a)**2 + 2440.2*(c0a + c1a)*(2*c0a/(c0a + c1a)**3 - 2/(c0a + c1a)**2) + 2440.2/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c0a/(c0a + c1a)**2 + 1/(c0a + c1a))/c0a, -2440.2*c0a/(c0a + c1a)**2 + 2440.2*c1a/(c0a + c1a)**2 + 2440.2*(c0a + c1a)*(2*c0a/(c0a + c1a)**3 - 1/(c0a + c1a)**2) - 2440.2/(c0a + c1a)], [0, -2440.2*c0a/(c0a + c1a)**2 + 2440.2*c1a/(c0a + c1a)**2 + 2440.2*(c0a + c1a)*(2*c0a/(c0a + c1a)**3 - 1/(c0a + c1a)**2) - 2440.2/(c0a + c1a), 2440.2*c0a/(c0a + c1a)**2 - 2440.2*c1a/(c0a + c1a)**2 + 2440.2*(c0a + c1a)*(2*c1a/(c0a + c1a)**3 - 2/(c0a + c1a)**2) + 2440.2/(c0a

In [5]:
x_values = np.linspace(0.001, .999, 100)
ya = fa(c0a=x_values, c1a=1-x_values, Va=1)
yb = fb(c0b=x_values, c1b=1-x_values, Vb=1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_values, y=ya, mode='lines', name='fa'))
fig.add_trace(go.Scatter(x=x_values, y=yb, mode='lines', name='fb'))
fig.update_layout(
    xaxis_title='x',
    yaxis_title='Energy',
    title='fa and fb',
    legend_title='Function'
)
fig.show()

# Combine functions

We now combine both functions into a composite pytential, and add constraints for the total of each species. 

Note the pytential takes c0 and c1 as arguments even though they only appear in the constraints. This implies applicability to infeasible regions which could be explored if the constraints were implemented as barrier methods.

In [6]:
f = fa+fb

# Define and add constraints
c0, c1 = symbols('c0, c1')
f = f.add_constraints_sym([c0a+c0b-c0, c1a+c1b-c1]) 

# Read expression
print(f)

x = ['Va', 'Vb', 'c0', 'c0a', 'c0b', 'c1', 'c1a', 'c1b']

f(x) = 2440.2*c0a*(log(c0a/(c0a + c1a)) + 2) + 2440.2*c0b*log(c0b/(c0b + c1b)) + 2440.2*c1a*log(c1a/(c0a + c1a)) + 2440.2*c1b*(log(c1b/(c0b + c1b)) + 1)

f'(x)= [0, 0, 0, -2440.2*c1a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c0a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 2440.2*log(c0a/(c0a + c1a)) + 4880.4, -2440.2*c1b/(c0b + c1b) + 2440.2*(c0b + c1b)*(-c0b/(c0b + c1b)**2 + 1/(c0b + c1b)) + 2440.2*log(c0b/(c0b + c1b)), 0, -2440.2*c0a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c1a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 2440.2*log(c1a/(c0a + c1a)), -2440.2*c0b/(c0b + c1b) + 2440.2*(c0b + c1b)*(-c1b/(c0b + c1b)**2 + 1/(c0b + c1b)) + 2440.2*log(c1b/(c0b + c1b)) + 2440.2]

f"(x)= [[0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, -2440.2*c0a/(c0a + c1a)**2 + 2440.2*c1a/(c0a + c1a)**2 + 2440.2*(c0a + c1a)*(2*c0a/(c0a + c1a)**3 - 2/(c0a + c1a)**2) + 2440.2/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c0a/(c0a + c1a)**2 + 1/(c0a + c1a))

# Equilibrium states

## Zero pressure

Given a particular composition, we want to find the equilibrium (minimum) energy of the total system. To do this, we define a new pytential from $f$ which is a function of the overall composition and determine the remaining arguments by minimization. 

In [7]:
f_min = min_pytential(f.add_constraints_sym([Va+Vb-1]), ['c0'])

In [8]:
# f_min is no longer a sympy expression, so printing it references the functions 
# used to determine the result. Note the constraints have been incorporated and are
# therefore no longer explicitly present. 
print(f_min)

cont []
Pytential of type <class 'pytential.reduce.min_pytential.min_pytential'>
Variables: ['c0']
Potential: <bound method args_to_list.<locals>.wrapper of <pytential.reduce.min_pytential.min_pytential object at 0x000002C5E450FE00>>
Gradient: <bound method args_to_list.<locals>.wrapper of <pytential.reduce.min_pytential.min_pytential object at 0x000002C5E450FE00>>
Hessian: <bound method args_to_list.<locals>.wrapper of <pytential.reduce.min_pytential.min_pytential object at 0x000002C5E450FE00>>



In [9]:
ym = f_min(x_values)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_values, y=ya, mode='lines', name='fa'))
fig.add_trace(go.Scatter(x=x_values, y=yb, mode='lines', name='fb'))
fig.add_trace(go.Scatter(x=x_values, y=ym, mode='lines', name='f_min'))
fig.update_layout(
    xaxis_title='x',
    yaxis_title='Energy',
    title='fa, fb, and f_min',
    legend_title='Function'
)
fig.show()

The minimal energy partitions the overall composition, $c_0$, between phases subject to $c_0=c_0^a+c_0^b$, and similarly for $c_1$. We can find the equilibrium partition:

In [10]:
print(f_min.min_fcn(np.array([[.4]]))[1][0])

{'c0': 0.4, 'Va': 0.46111634923273426, 'Vb': 0.5388836507672657, 'c0a': 0.041514480400231035, 'c0b': 0.358485519599769, 'c1': 0.6, 'c1a': 0.4196018688325032, 'c1b': 0.18039813116749673}


Graphically, partitioning defines the lowest common tangent between $f^a$ and $f^b$ seen above while the volume of each phase, $V^a$ and $V^b$ goes from one to zero.

## Controlling V

In [11]:
f2 = f.add_constraints_sym([c0+c1-1, Va+Vb-1])
f_min2 = min_pytential(f2, ['c0', 'Va'])

In [ ]:
import numpy as np
import plotly.graph_objects as go

x_values = np.linspace(0.001, .999, 10)

#RESHAPE meshgrid and use the broadcasting rules to evaluate instead of this function. 

# Create a meshgrid for the two arguments
X, Y = np.meshgrid(x_values, x_values)

# Evaluate f_min2 for each pair of (c0, Va)
F_min2 = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        F_min2[i, j] = f_min2(c0=X[i, j], Va=Y[i, j])

# Create a Plotly surface plot
fig = go.Figure(data=[go.Surface(z=F_min2, x=X, y=Y, colorscale='Viridis')])

# Add a line plot for f_a at Va = 1
fa_values_Va_1 = fa(c0a=x_values, c1a=1-x_values, Va=1)
fig.add_trace(go.Scatter3d(
    x=x_values,
    y=[1] * len(x_values),  # Va = 1
    z=fa_values_Va_1,
    mode='lines',
    name='f_a at Va=1',
    line=dict(color='blue')
))

# Add a line plot for f_b at Va = 0
fb_values_Va_0 = fb(c0b=x_values, c1b=1-x_values, Vb=1)
fig.add_trace(go.Scatter3d(
    x=x_values,
    y=[0] * len(x_values),  # Va = 0
    z=fb_values_Va_0,
    mode='lines',
    name='f_b at Va=0',
    line=dict(color='green')
))


# Add labels and title
fig.update_layout(
    title="Surface Plot of f_min2",
    scene=dict(
        xaxis_title="c0",
        yaxis_title="Va",
        zaxis_title="f_min2",
    ),
)

# Show the plot
fig.show()

C:\Users\wellandm\AppData\Local\Temp\ipykernel_15808\853898238.py:15: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

C:\Users\wellandm\AppData\Local\Temp\ipykernel_15808\853898238.py:15: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

C:\Users\wellandm\AppData\Local\Temp\ipykernel_15808\853898238.py:15: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

C:\Users\wellandm\AppData\Local\Temp\ipykernel_15808\853898238.py:15: DeprecationWarning:

Conversion of an array with nd